# Regresión Lineal y Redes Neuronales Artificiales — Ejemplos

**Módulo 4 — Modelado y evaluación de enfoques supervisados · Curso Analítica de Datos**

Este notebook acompaña las diapositivas [`4.8_Regresion_Lineal.pdf`](4.8_Regresion_Lineal.pdf) y [`4.7_Redes_Neuronales_Artificiales.pdf`](4.7_Redes_Neuronales_Artificiales.pdf), y lleva a código lo que allí se explica.

## Contenido

**Parte 1 — Regresión lineal**
1.1 Desde cero: descenso de gradiente sobre la función de costo $J(w,b)$, con el ejemplo de ingreso vs. años de educación de las diapositivas.
1.2 Con `scikit-learn`: regresión lineal múltiple sobre datos clínicos reales (*Diabetes dataset*).
1.3 Regresión polinomial: cómo el grado del polinomio cambia la flexibilidad del modelo.
1.4 Regularización: reproducimos el ejemplo exacto de sobre-entrenamiento de las diapositivas ($y=x^2$) y lo corregimos con Ridge y Lasso.

**Parte 2 — Redes neuronales artificiales**
2.1 Desde cero: funciones de activación (sigmoid, tanh, ReLU) y el cálculo manual de una neurona.
2.2 Con `scikit-learn`: una red neuronal (`MLPClassifier`) sobre un problema no lineal, mostrando el efecto de la regularización (`alpha`) en el sobreajuste.

> 💡 Este notebook **no requiere descargar nada ni tener cuenta de Kaggle**: el *Diabetes dataset* viene incluido en `scikit-learn`, y los demás ejemplos usan datos sintéticos generados en el propio notebook.

In [ ]:
# Librerías que usaremos en todo el notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes, make_moons
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import r2_score

pd.set_option('display.max_columns', None)
rng = np.random.default_rng(42)

## 1. Regresión lineal

### 1.1 Desde cero: descenso de gradiente

Reproducimos el ejemplo de las diapositivas: el **ingreso** de una persona (en miles de USD/año) en función de sus **años de educación**. El modelo es $\hat{y} = wx + b$, y buscamos los valores de $w$ (pendiente) y $b$ (intercepto) que minimizan la función de costo:

$$J(w,b) = \frac{1}{2m}\sum_{i=0}^{m-1}\left[f_{w,b}(x^{(i)}) - y^{(i)}\right]^2$$

exactamente como aparece en las diapositivas.

In [ ]:
# Datos sintéticos: anios de educacion -> ingreso anual (miles de USD)
n = 60
educacion = rng.uniform(0, 20, n)
salario = 20 + 2.5 * educacion + rng.normal(0, 5, n)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(educacion, salario, alpha=0.6)
ax.set_xlabel('Años de educación')
ax.set_ylabel('Ingreso anual (miles USD)')
plt.tight_layout()
plt.show()

In [ ]:
def costo(w, b, x, y):
    """J(w, b), exactamente como en las diapositivas."""
    m = len(x)
    return (1 / (2 * m)) * np.sum((w * x + b - y) ** 2)

def gradiente(w, b, x, y):
    m = len(x)
    pred = w * x + b
    dw = (1 / m) * np.sum((pred - y) * x)
    db = (1 / m) * np.sum(pred - y)
    return dw, db

def descenso_gradiente(x, y, lr=0.005, epochs=20000):
    w, b = 0.0, 0.0
    historia = [(w, b, costo(w, b, x, y))]
    for _ in range(epochs):
        dw, db = gradiente(w, b, x, y)
        w -= lr * dw
        b -= lr * db
        historia.append((w, b, costo(w, b, x, y)))
    return w, b, historia

w_final, b_final, historia = descenso_gradiente(educacion, salario)
print(f'Descenso de gradiente:    w={w_final:.3f}, b={b_final:.3f}')

# Verificamos contra la solucion de scikit-learn
reg_sk = LinearRegression().fit(educacion.reshape(-1, 1), salario)
print(f'scikit-learn LinearRegression: w={reg_sk.coef_[0]:.3f}, b={reg_sk.intercept_:.3f}')

Según el modelo, **cada año adicional de educación se asocia a un incremento de ~2.5 miles de dólares anuales** ($\beta_1$), y una persona con 0 años de educación tendría un ingreso base de ~19 mil dólares ($\beta_0$) —tal como explican las diapositivas con el ejemplo del salario mínimo.

Visualicemos la función de costo $J(w,b)$ (la "tazón" de las diapositivas) y el camino que recorrió el descenso de gradiente hasta encontrar el mínimo.

In [ ]:
ws_hist = [h[0] for h in historia]
bs_hist = [h[1] for h in historia]

ww = np.linspace(0, 5, 60)
bb = np.linspace(0, 40, 60)
WW, BB = np.meshgrid(ww, bb)
JJ = np.array([[costo(wv, bv, educacion, salario) for wv in ww] for bv in bb])

fig = plt.figure(figsize=(11, 4.5))

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(WW, BB, JJ, cmap='viridis', alpha=0.7)
ax1.set_xlabel('w'); ax1.set_ylabel('b'); ax1.set_zlabel('J(w, b)')
ax1.set_title('Función de costo J(w, b)')

ax2 = fig.add_subplot(1, 2, 2)
ax2.contour(WW, BB, JJ, levels=30, cmap='viridis')
ax2.plot(ws_hist, bs_hist, 'r.-', markersize=2)
ax2.scatter([w_final], [b_final], c='red', marker='*', s=200, zorder=5)
ax2.set_xlabel('w'); ax2.set_ylabel('b')
ax2.set_title('Camino del descenso de gradiente')

plt.tight_layout()
plt.show()

La trayectoria roja muestra cómo, partiendo de $w=0, b=0$, cada iteración del descenso de gradiente da un paso en la dirección que más reduce $J(w,b)$, hasta llegar al mínimo de la función (la estrella). Grafiquemos finalmente la recta de regresión resultante sobre los datos.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(educacion, salario, alpha=0.6, label='Datos observados')
xx = np.linspace(0, 20, 50)
ax.plot(xx, w_final * xx + b_final, 'r-', linewidth=2, label='Recta de regresión')
ax.set_xlabel('Años de educación')
ax.set_ylabel('Ingreso anual (miles USD)')
ax.legend()
plt.tight_layout()
plt.show()

### 1.2 Con `scikit-learn`: regresión lineal múltiple

Ahora un ejemplo con datos reales y varias variables: el **Diabetes dataset** de `scikit-learn` contiene 10 variables clínicas (edad, sexo, índice de masa corporal, presión arterial y seis mediciones de sangre) de 442 pacientes, y como variable objetivo una medida cuantitativa de **progresión de la enfermedad** un año después del diagnóstico.

In [ ]:
diabetes = load_diabetes()
Xdi, ydi = diabetes.data, diabetes.target
print(f'{Xdi.shape[0]} pacientes, {Xdi.shape[1]} variables clínicas: {list(diabetes.feature_names)}')

Xdi_tr, Xdi_te, ydi_tr, ydi_te = train_test_split(Xdi, ydi, test_size=0.3, random_state=42)

reg_diabetes = LinearRegression().fit(Xdi_tr, ydi_tr)
ydi_pred = reg_diabetes.predict(Xdi_te)
print(f'R² en el conjunto de prueba: {r2_score(ydi_te, ydi_pred):.3f}')

coefs = pd.Series(reg_diabetes.coef_, index=diabetes.feature_names).sort_values(key=abs, ascending=False)
print('\nVariables con mayor peso en la predicción:')
print(coefs.head(5).round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(ydi_te, ydi_pred, alpha=0.6)
lims = [min(ydi_te.min(), ydi_pred.min()), max(ydi_te.max(), ydi_pred.max())]
ax.plot(lims, lims, 'k--', label='Predicción perfecta')
ax.set_xlabel('Progresión real de la enfermedad')
ax.set_ylabel('Progresión predicha')
ax.legend()
plt.tight_layout()
plt.show()

Los puntos no caen exactamente sobre la diagonal ($R^2 \approx 0.48$): el modelo captura una tendencia real, pero con margen de error considerable — típico en datos clínicos con alta variabilidad entre pacientes.

### 1.3 Regresión polinomial: ¿qué tan flexible debe ser el modelo?

Las diapositivas muestran que una regresión lineal **siempre produce una tendencia lineal** — pero usando **funciones base polinomiales** ($\phi_m(x) = x^m$) podemos ajustar tendencias no lineales. Reproducimos el ejemplo de las diapositivas: ajustar $y = \sin(2x)$ con polinomios de distinto grado.

In [ ]:
xr = np.linspace(-3, 3, 80)
yr = np.sin(2 * xr) + rng.normal(0, 0.2, len(xr))
Xr = xr.reshape(-1, 1)

grados = [1, 3, 5, 11]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))

for ax, grado in zip(axes, grados):
    modelo = make_pipeline(PolynomialFeatures(grado), LinearRegression()).fit(Xr, yr)
    x_plot = np.linspace(-3, 3, 300).reshape(-1, 1)
    y_plot = modelo.predict(x_plot)
    ax.scatter(xr, yr, s=10, alpha=0.5, label='Muestras')
    ax.plot(x_plot, np.sin(2 * x_plot), 'r-', label='y=sin(2x)')
    ax.plot(x_plot, y_plot, 'b-', label='Ajuste')
    ax.set_title(f'grado={grado}')
    ax.set_ylim(-2, 2)

axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

Con **grado 1** (una línea recta) el modelo es demasiado simple para capturar la curva — *underfitting*. A medida que aumentamos el grado, el polinomio tiene más libertad para seguir la curva real, y con **grado 11** el ajuste es prácticamente perfecto sobre los datos de entrenamiento.

### 1.4 Regularización: corrigiendo el sobreajuste

¿Qué pasa si le damos a un polinomio **demasiada** libertad con **pocos** datos? Reproducimos el ejemplo exacto de sobre-entrenamiento de las diapositivas: ajustar $y = x^2$ con un polinomio de grado alto sobre solo 12 observaciones ruidosas.

In [ ]:
rng_overfit = np.random.default_rng(3)  # semilla propia, fija, para que el ejemplo sea reproducible
xo = np.sort(rng_overfit.uniform(0, 2, 12))
yo = xo ** 2 + rng_overfit.normal(0, 0.25, 12)

grado_alto = 9
poly = PolynomialFeatures(grado_alto)
Xo_poly = poly.fit_transform(xo.reshape(-1, 1))

modelos_reg = {
    'Sin regularizar (grado 9)': LinearRegression(),
    'Ridge (alpha=5)': Ridge(alpha=5),
    'Lasso (alpha=0.05)': Lasso(alpha=0.05, max_iter=50000),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
coef_counts = {}
for ax, (nombre, modelo) in zip(axes, modelos_reg.items()):
    modelo.fit(Xo_poly, yo)
    x_plot = np.linspace(-0.1, 2.1, 300).reshape(-1, 1)
    y_plot = modelo.predict(poly.transform(x_plot))
    ax.scatter(xo, yo, s=20)
    ax.plot(x_plot, x_plot ** 2, 'r-', label='y=x² (real)')
    ax.plot(x_plot, y_plot, 'b-', label='Ajuste')
    ax.set_title(nombre)
    ax.set_ylim(-2, 5)
    ax.legend(fontsize=7)
    coef_counts[nombre] = int(np.sum(np.abs(modelo.coef_) > 1e-3))

plt.tight_layout()
plt.show()
print('Coeficientes distintos de cero por modelo:', coef_counts)

**Sin regularizar**, el polinomio de grado 9 "memoriza" las 12 muestras y oscila violentamente entre ellas —exactamente lo que muestran las diapositivas con el ejemplo de $y=x^2$—. **Ridge** y **Lasso** penalizan pesos grandes y producen un ajuste mucho más suave y razonable. Además, **Lasso** reduce varios coeficientes exactamente a cero (de 9 a solo 3 activos): hace *selección de variables* además de regularizar, tal como explican las diapositivas.

Nótese que incluso los modelos regularizados se vuelven inciertos justo en el borde derecho, donde casi no hay datos — la regularización ayuda a controlar el sobreajuste *dentro* del rango de los datos, pero extrapolar más allá de ellos siempre es arriesgado.

## 2. Redes neuronales artificiales

### 2.1 Desde cero: funciones de activación y una neurona

Las diapositivas muestran que cada neurona de una red calcula una combinación lineal de sus entradas y luego le aplica una **función de activación** $f(\cdot)$. Implementamos las tres funciones de activación más comunes.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def relu(x):
    return np.maximum(0, x)

xx_act = np.linspace(-6, 6, 200)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, f, nombre in zip(axes, [sigmoid, tanh, relu], ['Sigmoid', 'Tanh', 'ReLU']):
    ax.plot(xx_act, f(xx_act))
    ax.set_title(nombre)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Una neurona, replicando la notacion de las diapositivas:
# l11 = f1(x1*w1,1 + x2*w2,1 + x3*w3,1 + x4*w4,1 + b11)
x_in = np.array([0.5, -0.2, 0.1, 0.8])      # cuatro entradas (capa de entrada)
w_11 = np.array([0.3, -0.5, 0.2, 0.7])      # pesos hacia la primera neurona oculta
b_11 = 0.1

combinacion_lineal = np.dot(x_in, w_11) + b_11
l11 = sigmoid(combinacion_lineal)

print(f'Combinación lineal (antes de activar): {combinacion_lineal:.4f}')
print(f'Salida de la neurona l11 (después de sigmoid): {l11:.4f}')

La neurona recibe las 4 entradas, las combina con sus pesos y su sesgo ($b_{11}$), y pasa el resultado por la función sigmoid para producir un valor entre 0 y 1. Una red completa repite este cálculo capa por capa —de ahí las fórmulas $l_{11}, l_{12}, l_{13}, \ldots$ de las diapositivas— hasta llegar a la capa de salida.

### 2.2 Con `scikit-learn`: una red neuronal y el efecto de la regularización

Entrenamos un `MLPClassifier` (*Multi-Layer Perceptron*) sobre un problema que **no es linealmente separable**: dos "lunas" entrelazadas. Las diapositivas advierten que las redes neuronales son propensas al sobreajuste, y que se suele usar **regularización** (el hiperparámetro `alpha`) para mitigarlo — veámoslo en acción, reproduciendo el espíritu de la cuadrícula de la diapositiva "Neural Networks - Comentarios".

In [ ]:
X_moons, y_moons = make_moons(n_samples=200, noise=0.25, random_state=42)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(X_moons, y_moons, test_size=0.3, random_state=42, stratify=y_moons)

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='bwr', s=15, edgecolors='k', linewidths=0.3)
ax.set_title('Dos "lunas" entrelazadas (no separables linealmente)')
plt.tight_layout()
plt.show()

In [ ]:
alphas = [0.0001, 0.01, 1.0, 10.0]
xx, yy = np.meshgrid(np.linspace(X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5, 200),
                      np.linspace(X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5, 200))

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, alpha in zip(axes, alphas):
    mlp = MLPClassifier(hidden_layer_sizes=(20, 10), alpha=alpha, max_iter=3000, random_state=0).fit(Xm_tr, ym_tr)
    acc_tr = mlp.score(Xm_tr, ym_tr)
    acc_te = mlp.score(Xm_te, ym_te)
    Z = mlp.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='bwr', s=10, edgecolors='k', linewidths=0.2)
    ax.set_title(f'alpha={alpha}\ntrain={acc_tr:.2f}  test={acc_te:.2f}')

plt.tight_layout()
plt.show()

Con `alpha` muy **pequeño** (poca regularización), la red tiene libertad para trazar una frontera de decisión que se ajusta muy de cerca a los datos de entrenamiento. Con `alpha` muy **grande** (mucha regularización), los pesos de la red se ven forzados a ser pequeños, y la frontera se vuelve casi una línea recta —la red pierde la capacidad de capturar la forma curva real de las dos lunas (*underfitting*)—. Como en los ejemplos anteriores de regularización, el valor adecuado de `alpha` está en un punto intermedio, y se elige empíricamente usando un conjunto de validación.

---

## Actividad abierta para discutir en clase

1. En la sección 1.1, cambien la semilla de los datos sintéticos (`rng = np.random.default_rng(...)` con otro número) y vuelvan a correr el descenso de gradiente. ¿Los valores de $w$ y $b$ cambian mucho? ¿Por qué el descenso de gradiente de todas formas converge a (casi) lo mismo que `scikit-learn` sin importar la semilla?
2. En la sección 1.4, aumenten aún más `alpha` en Ridge y Lasso (por ejemplo, a 50 o 500). ¿En qué momento el modelo deja de seguir la tendencia real de los datos? Relacionen esto con la frase de las diapositivas: "entre mayor sea el valor de $\lambda$, más fuerte se penaliza la función de error".
3. En la sección 2.1, calculen a mano (o con una celda de código) la salida de una **segunda** neurona oculta con otro vector de pesos $w_{12}$ y compárenla con $l_{11}$.
4. En la sección 2.2, ¿por qué una regresión lineal (sección 1) nunca podría separar bien las dos lunas, sin importar cuánto la regularicen? ¿Qué tienen las redes neuronales (o los árboles/SVM con kernel del notebook anterior) que las regresiones lineales simples no tienen?